# 收益与波动率预测实验

比较螺纹钢（RB）和铜（CU）在三种采样轴、三种季节性处理下的 ARMA(1,1)-GARCH(1,1) 收益与条件方差预测，模型含均值项。

采样轴为时间（time）、全合约成交额（money）和主力合约Garman-Klass波动率×全合约成交额（gk_money）。N=15表示采样单位为校准训练期平均分钟轴增量的15倍；log_interpolate按对数价格插值。观测处理为raw（不去季节）、TD120-RV（120交易日实现方差季节曲线）和TD120-BPV（120交易日双幂变差季节曲线）。2品种×3轴×3处理构成18组配置。

读取与复权范围为2010-01-01—2026-06-30，采样尺度校准训练期为2010-01-01—2024-12-31，测试期为2025-01-01—2026-06-30。三个计算日期为2025-01-02、2025-09-25、2026-06-30，分别对应固定359个测试交易日的位置0、179、358；时区为Asia/Shanghai。每个测试日使用此前完整三个自然年的历史拟合一次，日内先预测、再用观测更新状态，参数不重新估计。

54条拟合和1,017个完整预测位置已验收：30条拟合成功、24条失败，619个预测位置成功、398个失败。结果仅覆盖这18组配置和三个测试日，原规划1,944组配置未整体完成。

顺序运行只读取、核对和展示已保存结果。文件用途与复现方式见 [README.md](README.md)，执行身份与验收记录见 [provenance.json](provenance.json)。


In [1]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
project_root = next((candidate_root for candidate_root in (Path.cwd(), *Path.cwd().parents)
                     if (candidate_root/".env.template").is_file()
                     and (candidate_root/"R04_Research").is_dir()), None)
if project_root is None:
    raise FileNotFoundError("请从仓库目录启动Notebook；根定位使用.env.template")
experiment_dir = project_root/"R04_Research/a02_Experiments/b01_ReturnVolatilityForecasting"
sys.path[:0] = [str(project_root)]
from R04_Research.a00_notebook_loader import load_notebook_exports
load_notebook_exports(experiment_dir/"experiment.py", globals())


In [2]:

from pathlib import Path
import json
import hashlib
import base64
import zipfile
import sys
import shutil
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf


def load_experiment_config(config_dir):
    """组合最终配置；不认证、不提交、不计算。"""
    with initialize_config_dir(config_dir=str(Path(config_dir).resolve()), version_base="1.3"):
        config = compose(config_name="config")
    return config, sorted(OmegaConf.missing_keys(config))


def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8*1024*1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _delivery_path(directory, relative):
    root = Path(directory).resolve()
    path = (root/relative).resolve()
    if not path.is_relative_to(root):
        raise ValueError("交付路径越出本实验")
    return path


def read_delivery(experiment_dir, *, verify_inputs=False):
    """复读已验收四张成果表，核对摘要、Schema、范围和失败；默认不读大输入表。"""
    root = Path(experiment_dir).resolve()
    provenance = json.loads((root/"provenance.json").read_text(encoding="utf8"))
    if provenance["state"] != "accepted_final_delivery" or provenance["execution"]["allow_new_submission"]:
        raise ValueError("交付状态或执行禁止标志不正确")
    tables = {}
    for group in ("primary_tables","previous_tables"):
        for kind, info in provenance[group].items():
            path = _delivery_path(root, info["path"])
            schema = pa.ipc.read_schema(pa.BufferReader(base64.b64decode(info["schema_base64"])))
            if _sha256(path) != info["sha256"]:
                raise ValueError("成果 SHA 不一致："+info["path"])
            table = pq.read_table(path)
            if table.num_rows != info["rows"] or not table.schema.equals(schema,check_metadata=False):
                raise ValueError("成果 Schema／行数不一致："+info["path"])
            tables[("previous_" if group == "previous_tables" else "")+kind] = table.cast(schema,safe=True)
    fit, prediction = tables["fit"],tables["prediction"]
    if (fit.num_rows,prediction.num_rows)!=(54,1017):
        raise ValueError("18配置54拟合1017预测位置覆盖不一致")
    if (sum(s!="ok" for s in fit["status"].to_pylist()),
            sum(s!="ok" for s in prediction["status"].to_pylist())) != (24,398):
        raise ValueError("正式失败记录数量变化")
    configuration_ids = {item["task"]["task_id"] for item in provenance["configuration_results"]}
    if set(fit["configuration_id"].to_pylist()) != configuration_ids or len(configuration_ids)!=18:
        raise ValueError("配置身份覆盖不同")
    fit_rows = fit.to_pylist()
    for item in provenance["configuration_results"]:
        task = item["task"]
        rows = [row for row in fit_rows if row["configuration_id"]==task["task_id"]]
        if sorted(row["experiment_date"].isoformat() for row in rows)!=sorted(provenance["selection"]["dates"]):
            raise ValueError("每配置固定三日覆盖不同")
        if any((row["underlying_code"],row["axis"],row["N"],row["train_years"],row["case_id"])
               !=(task["underlying_code"],task["axis"],15,3,task["model_case"]["id"]) for row in rows):
            raise ValueError("成果参数身份不同")
    points = [(row["configuration_id"],row["experiment_date"],row["sample_id"]) for row in prediction.to_pylist()]
    if len(points)!=len(set(points)):
        raise ValueError("完整预测位置重复")
    source = provenance["source_package"]
    if _sha256(_delivery_path(root,source["path"]))!=source["sha256"]:
        raise ValueError("原云源码包摘要不同")
    if verify_inputs:
        for variety, record in provenance["inputs"].items():
            manifest = record["original_manifest"]
            body = {key:value for key,value in manifest.items() if key!="input_id"}
            if hashlib.sha256(json.dumps(body,sort_keys=True).encode()).hexdigest()!=manifest["input_id"]:
                raise ValueError("冻结输入身份不同："+variety)
            for name, info in record["files"].items():
                path = _delivery_path(root,info["path"])
                parquet = pq.ParquetFile(path)
                schema = pa.ipc.read_schema(pa.BufferReader(base64.b64decode(info["schema_base64"])))
                if (_sha256(path)!=info["sha256"] or parquet.metadata.num_rows!=info["rows"]
                        or not parquet.schema_arrow.equals(schema,check_metadata=True)):
                    raise ValueError("冻结输入发生变化："+info["path"])
        with zipfile.ZipFile(_delivery_path(root,source["path"])) as archive:
            for name,digest in source["original_cloud_code_manifest"]["files"].items():
                if hashlib.sha256(archive.read(name)).hexdigest()!=digest:
                    raise ValueError("原云代码文件变化："+name)
    return provenance,tables


RESULT_LABELS = {
 "underlying_code":"underlying_code（品种）","axis":"axis（采样轴）","N":"N（等效分钟）",
 "train_years":"train_years（训练年数）","case_id":"case_id（模型与季节配置）",
 "fit_rows":"fit_rows（拟合记录数）","failed_fit_rows":"failed_fit_rows（失败拟合数）",
 "prediction_rows":"prediction_rows（预测位置数）","failed_prediction_rows":"failed_prediction_rows（失败预测数）",
 "experiment_date":"experiment_date（实验交易日）","training_samples":"training_samples（训练样本数）",
 "test_samples":"test_samples（测试位置数）","status":"status（状态）","reason":"reason（原因）",
 "sample_id":"sample_id（采样位置编号）","sample_at":"sample_at（采样时刻）","bar_at":"bar_at（观测可得时刻）",
 "observed_model_input":"observed_model_input（模型观测）","mean_forecast":"mean_forecast（均值预测）",
 "configuration_id":"configuration_id（配置身份）",
}


def delivery_summary(provenance):
    """十八配置汇总，表头与索引带中文；不重算。"""
    rows=[]
    for item in provenance["configuration_results"]:
        task=item["task"]
        rows.append({**{key:task[key] for key in ("underlying_code","axis","N","train_years")},
                     "case_id":task["model_case"]["id"],
                     **{key:item[key] for key in ("fit_rows","failed_fit_rows","prediction_rows","failed_prediction_rows")}})
    frame=pd.DataFrame(rows).rename(columns=RESULT_LABELS)
    frame.index.name="configuration_number（配置序号）"
    return frame


def result_rows(table, columns):
    """仅展示指定字段的副本，不改变正式表。"""
    if any(name not in RESULT_LABELS for name in columns):
        raise ValueError("展示字段须有明确中文名称")
    frame=table.select(columns).to_pandas().rename(columns=RESULT_LABELS)
    frame.index.name="row_number（行号）"
    return frame


def continue_experiment_notebook(*args, **kwargs):
    """完成的一次性实验禁止重新提交或恢复原1944队列；在任何I/O前停止。"""
    raise RuntimeError("实验已完成；禁止提交、重新启动或恢复原1944配置队列")


start_reduced_experiment_notebook = continue_experiment_notebook
submit_reduced_experiment_job = continue_experiment_notebook
submit_experiment_cloud_task = continue_experiment_notebook


def reproduce_day_locally(experiment_dir, *, work_dir, variety, axis, case_id, experiment_date):
    """显式在新的外部工作目录复现一个固定实验日；只调用冻结Notebook，不调用云API。

    此函数默认不执行。work_dir必须不存在或为空，且位于已完成实验目录之外；
    生成的工作副本与结果归调用者选择的目录，函数不删除任何文件。
    """
    root=Path(experiment_dir).resolve()
    work=Path(work_dir).resolve()
    if work.is_relative_to(root) or root.is_relative_to(work):
        raise ValueError("复现目录必须与完成交付目录分离")
    if work.exists() and any(work.iterdir()):
        raise FileExistsError("复现工作目录必须为空")
    provenance=json.loads((root/"provenance.json").read_text(encoding="utf8"))
    matched=[item["task"] for item in provenance["configuration_results"]
             if item["task"]["underlying_code"]==variety and item["task"]["axis"]==axis
             and item["task"]["model_case"]["id"]==case_id]
    if len(matched)!=1 or experiment_date not in provenance["selection"]["dates"]:
        raise ValueError("只复现已确定的十八配置及三个日期")
    package=provenance["source_package"]
    archive_path=_delivery_path(root,package["path"])
    if _sha256(archive_path)!=package["sha256"]:
        raise ValueError("冻结代码包摘要不同")
    work.mkdir(parents=True,exist_ok=True)
    source=work/"source"
    with zipfile.ZipFile(archive_path) as archive:
        for entry in archive.infolist():
            if not (source/entry.filename).resolve().is_relative_to(source.resolve()):
                raise ValueError("源码包路径越界")
        archive.extractall(source)
    frozen_input=work/"input"
    frozen_input.mkdir()
    input_record=provenance["inputs"][variety]
    for name,info in input_record["files"].items():
        original=_delivery_path(root,info["path"])
        if _sha256(original)!=info["sha256"]:
            raise ValueError("冻结输入不同")
        shutil.copyfile(original,frozen_input/name)
    (frozen_input/"manifest.json").write_text(
        json.dumps(input_record["original_manifest"],ensure_ascii=False,sort_keys=True,indent=2)+"\n",encoding="utf8")
    from R04_Research.a00_notebook_loader import load_notebook_exports
    old_path=list(sys.path)
    try:
        sys.path[:0]=[str(source)]
        namespace={"__name__":"_rvf_frozen_numerical_notebook"}
        loader_path=source/"R04_Research/a02_Experiments/b01_ReturnVolatilityForecasting/experiment.py"
        load_notebook_exports(loader_path,namespace)
        task={key:value for key,value in matched[0].items() if key!="task_id"}
        task["dates"]=[experiment_date]
        task["task_id"]=hashlib.sha256(json.dumps(task,sort_keys=True).encode()).hexdigest()
        return namespace["run_experiment_task"](frozen_input,task,work/"result")
    finally:
        sys.path[:]=old_path


## 读取与配置汇总

读取 `config.yaml` 和 `provenance.json`，核对结果表的SHA256、Schema、行数及配置身份。下表按品种、采样轴和观测处理列出全部18组配置；失败记录参与覆盖统计，不能据此视为有效预测。


In [3]:
experiment_config, missing = load_experiment_config(experiment_dir)
if missing:
    raise ValueError(missing)
provenance, tables = read_delivery(experiment_dir)
display(delivery_summary(provenance))


,underlying_code（品种）,axis（采样轴）,N（等效分钟）,train_years（训练年数）,case_id（模型与季节配置）,fit_rows（拟合记录数）,failed_fit_rows（失败拟合数）,prediction_rows（预测位置数）,failed_prediction_rows（失败预测数）
configuration_number（配置序号）,,,,,,,,,
0,RB,time,15,3,arma_garch_no_seasonality,3,0,61,0
1,RB,time,15,3,arma_garch_td120_rv,3,2,61,46
2,RB,time,15,3,arma_garch_td120_bpv,3,3,61,61
3,RB,money,15,3,arma_garch_no_seasonality,3,0,19,0
4,RB,money,15,3,arma_garch_td120_rv,3,2,19,9
5,RB,money,15,3,arma_garch_td120_bpv,3,3,19,19
6,RB,gk_money,15,3,arma_garch_no_seasonality,3,0,13,0
7,RB,gk_money,15,3,arma_garch_td120_rv,3,2,13,4
8,RB,gk_money,15,3,arma_garch_td120_bpv,3,3,13,13


## 失败记录与预测位置

24条失败拟合的训练观测含非有限值，在正式输入检查处停止，优化器未执行。BPV历史的季节因素存在固定迭代Huber非收敛或零MAD缺失；RB的TD120-RV配置在2025-09-25和2026-06-30含零MAD缺失。基础设施失败为0。

拟合失败对应的398个预测位置均保留失败状态和原因；未换模型、补值、删除缺失观测或重试拟合。下面先列出全部失败拟合，再预览预测表前20行。完整位置及坐标保存在 `Data/prediction.parquet`，预览行数不代表实验覆盖范围。


In [4]:
display(result_rows(tables["fit"].filter(pc.not_equal(tables["fit"]["status"],"ok")),
                    ["experiment_date","underlying_code","axis","case_id","training_samples","status","reason"]))


,experiment_date（实验交易日）,underlying_code（品种）,axis（采样轴）,case_id（模型与季节配置）,training_samples（训练样本数）,status（状态）,reason（原因）
row_number（行号）,,,,,,,
0,2025-01-02,RB,gk_money,arma_garch_td120_bpv,10554,failed,observations 必须为非空有限一维序列；不自动删除缺失行
1,2025-09-25,RB,money,arma_garch_td120_bpv,10622,failed,observations 必须为非空有限一维序列；不自动删除缺失行
2,2026-06-30,RB,time,arma_garch_td120_bpv,16523,failed,observations 必须为非空有限一维序列；不自动删除缺失行
3,2025-09-25,RB,gk_money,arma_garch_td120_bpv,8524,failed,observations 必须为非空有限一维序列；不自动删除缺失行
4,2025-09-25,RB,time,arma_garch_td120_rv,16592,failed,observations 必须为非空有限一维序列；不自动删除缺失行
5,2026-06-30,RB,money,arma_garch_td120_bpv,8500,failed,observations 必须为非空有限一维序列；不自动删除缺失行
6,2025-09-25,RB,money,arma_garch_td120_rv,10622,failed,observations 必须为非空有限一维序列；不自动删除缺失行
7,2026-06-30,RB,gk_money,arma_garch_td120_bpv,6496,failed,observations 必须为非空有限一维序列；不自动删除缺失行
8,2026-06-30,RB,time,arma_garch_td120_rv,16523,failed,observations 必须为非空有限一维序列；不自动删除缺失行


In [5]:
display(result_rows(tables["prediction"].slice(0,20),
                    ["experiment_date","sample_id","sample_at","bar_at","observed_model_input","mean_forecast","status","reason"]))


,experiment_date（实验交易日）,sample_id（采样位置编号）,sample_at（采样时刻）,bar_at（观测可得时刻）,observed_model_input（模型观测）,mean_forecast（均值预测）,status（状态）,reason（原因）
row_number（行号）,,,,,,,,
0,2025-09-25,10624,2025-09-24 21:39:22.169466+08:00,2025-09-24 21:40:00+08:00,0.001181,-1.024346e-04,ok,
1,2025-09-25,10625,2025-09-25 09:00:51.013030+08:00,2025-09-25 09:01:00+08:00,-0.002795,3.691871e-05,ok,
2,2025-09-25,10626,2025-09-25 09:49:59.493746+08:00,2025-09-25 09:50:00+08:00,-0.001229,-1.796746e-04,ok,
3,2025-09-25,10627,2025-09-25 10:45:47.433116+08:00,2025-09-25 10:46:00+08:00,0.003725,-1.039917e-04,ok,
4,2025-09-25,10628,2025-09-25 13:30:36.971656+08:00,2025-09-25 13:31:00+08:00,0.000235,1.806992e-04,ok,
5,2025-09-25,10629,2025-09-25 14:06:29.092169+08:00,2025-09-25 14:07:00+08:00,0.000463,2.166066e-07,ok,
6,2025-01-02,10556,2025-01-02 09:00:23.385141+08:00,2025-01-02 09:01:00+08:00,-0.000938,NaN,failed,fit: observations 必须为非空有限一维序列；不自动删除缺失行
7,2025-01-02,10557,2025-01-02 09:00:46.770283+08:00,2025-01-02 09:01:00+08:00,-0.000938,NaN,failed,fit: observations 必须为非空有限一维序列；不自动删除缺失行
8,2025-01-02,10558,2025-01-02 09:02:02.227039+08:00,2025-01-02 09:03:00+08:00,-0.001037,NaN,failed,fit: observations 必须为非空有限一维序列；不自动删除缺失行


## 核对输入与本地复现

`sources.zip` 保存实际云作业使用的源码和固定依赖，`Data/` 保存RB、CU的14张原冻结输入。`provenance.json` 将配置和结果对应到原范围、输入、代码与Run身份；说明文字的修改不改写原执行来源。

`read_delivery(experiment_dir, verify_inputs=True)` 核对全部冻结输入和源码快照。显式调用 `reproduce_day_locally(experiment_dir, work_dir=新空工作目录, variety="RB", axis="time", case_id="arma_garch_no_seasonality", experiment_date="2025-01-02")` 可使用同一完整历史复现指定配置和日期；函数默认不执行，仅在指定的外部工作目录生成副本及本地结果。

复现使用新的Python 3.13内核和 `sources.zip` 中 `requirements.txt` 的固定依赖，具体版本及文件用途见 [README.md](README.md)。本Notebook的默认读取与展示不触发复现或云提交；原1,944组配置队列入口已关闭，监督保持PAUSED。
